In [2]:
from pathlib import Path
from typing import List

import pymupdf4llm
from src.models.document import Document, DocumentMetadata

def load_pdf_with_pymupdf4llm(file_path: Path) -> List[Document]:
    """
    Parses multi-column PDFs into clean markdown chunks per page 
    using layout heuristics. Zero GPU required.
    """
    pdf_path = Path(file_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    # page_chunks=True returns a list of dictionaries, one per page
    pages_data = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)
    documents: List[Document] = []

    for page in pages_data:
        text = page["text"]
        page_num = page["metadata"]["page_number"]
        
        if not text.strip():
            continue

        metadata = DocumentMetadata(
            source=pdf_path.name,
            file_type="pdf",
            page=page_num
        )
        
        documents.append(Document(page_content=text, metadata=metadata))
        
    return documents

In [3]:
docs = load_pdf_with_pymupdf4llm(file_path="../documents/pdfs/MacBook Air (13-inch, M5) - Tech Specs.pdf")

In [4]:
docs

[Document(page_content='5/28/26, 11:20 AM \n\nMacBook Air (13-inch, M5) - Tech Specs - Apple Support (IN) \n\n**Documentation** \n\n**==> picture [99 x 99] intentionally omitted <==**\n\n## **MacBook Air (13-inch, M5) - Tech Specs** \n\nYear introduced: 2026 \n\n## **Finish** \n\nSky Blue Silver Starlight Midnight \n\n## **Chip** \n\n## **Apple M5 chip** \n\n10-core CPU with 4 super cores and 6 efficiency cores \n\n8-core GPU, 10-core GPU Neural Accelerators Hardware-accelerated ray tracing 16-core Neural Engine 153GB/s memory bandwidth \n\n## **Media Engine** \n\nHardware-accelerated H.264, HEVC, ProRes and ProRes RAW \n\nVideo decode engine Video encode engine ProRes encode and decode engine AV1 decode \n\n## **Configurable to:** \n\nM5 with 10-core CPU and 10-core GPU \n\nhttps://support.apple.com/en-in/126320 \n\n1/7 \n\n', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section=None, chunk_id=None, chunk_index=None, parent_do

In [17]:
docs[0].page_content

'5/28/26, 11:20 AM \n\nMacBook Air (13-inch, M5) - Tech Specs - Apple Support (IN) \n\n**Documentation** \n\n**==> picture [99 x 99] intentionally omitted <==**\n\n## **MacBook Air (13-inch, M5) - Tech Specs** \n\nYear introduced: 2026 \n\n## **Finish** \n\nSky Blue Silver Starlight Midnight \n\n## **Chip** \n\n## **Apple M5 chip** \n\n10-core CPU with 4 super cores and 6 efficiency cores \n\n8-core GPU, 10-core GPU Neural Accelerators Hardware-accelerated ray tracing 16-core Neural Engine 153GB/s memory bandwidth \n\n## **Media Engine** \n\nHardware-accelerated H.264, HEVC, ProRes and ProRes RAW \n\nVideo decode engine Video encode engine ProRes encode and decode engine AV1 decode \n\n## **Configurable to:** \n\nM5 with 10-core CPU and 10-core GPU \n\nhttps://support.apple.com/en-in/126320 \n\n1/7 \n\n'

# Cleaning / Preprocessing

In [5]:
import re
from typing import List

from src.models.document import Document

class PDFTextCleaner: 
    def __init__(self): 
        pass

    def remove_headers_and_footers(self, text: str) -> str:
        """
        Remove repeated PDF header/footer noise.
        """

        # Remove timestamp lines
        text = re.sub(
            r"\d{1,2}/\d{1,2}/\d{2},\s+\d{1,2}:\d{2}\s+[AP]M",
            "",
            text
        )

        # Remove Apple Support page title lines
        text = re.sub(
            r"MacBook Air \(13-inch, M5\) - Tech Specs - Apple Support \(IN\)",
            "",
            text
        )

        # Remove URLs
        text = re.sub(
            r"https?://\S+",
            "",
            text
        )

        # Remove page counters like 1/7
        text = re.sub(
            r"\b\d+/\d+\b",
            "",
            text
        )

        return text


    def remove_image_placeholders(self, text: str) -> str:
        """
        Remove pymupdf4llm image placeholder text.
        """

        text = re.sub(
            r"\*\*==> picture.*?omitted <==\*\*",
            "",
            text
        )

        return text


    def fix_broken_words(self, text: str) -> str:
        """
        Fix common OCR-style broken words conservatively.
        """

        replacements = {
            "Con fig ure": "Configure",
            "En viron men tal": "Environmental",
            "Accessi bility": "Accessibility",
        }

        for broken, fixed in replacements.items():
            text = text.replace(broken, fixed)

        return text


    def normalize_whitespace(self, text: str) -> str:
        """
        Normalize whitespace while preserving markdown structure.
        """

        # Remove trailing spaces
        text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)

        # Replace excessive blank lines (3+ → 2)
        text = re.sub(r"\n{3,}", "\n\n", text)

        # Strip leading/trailing whitespace
        text = text.strip()

        return text
    
   
    def remove_footer_noise(self, text: str) -> str:
        """
        Remove remaining website footer artifacts specifically at 7th page.
        """

        footer_patterns = [
            r"\*\*Helpful\?\*\* Yes No",
            r"Support MacBook Air.*",
            r"Copyright © .*",
            r"Privacy Policy",
            r"Terms of Use",
            r"Sales Policy",
            r"Site Map",
            r"\bIndia\b",
        ]

        for pattern in footer_patterns:
            text = re.sub(pattern, "", text)

        return text


    def fix_footnote_artifacts(self, text: str) -> str:
        """
        Fix merged footnote/superscript artifacts conservatively.
        Examples:
            Storage1 -> Storage
            Power3 -> Power
        """

        text = re.sub(
            r"\b([A-Za-z]{5,})(\d{1,2})\b",
            r"\1",
            text
        )

        return text




    def clean_text(self, text: str) -> str:
        """
        Apply full text cleaning pipeline.
        """

        text = self.remove_headers_and_footers(text)
        text = self.remove_image_placeholders(text)
        text = self.fix_broken_words(text)
        text = self.normalize_whitespace(text)
        text = self.remove_footer_noise(text)
        text = self.fix_footnote_artifacts(text)

        return text


    def clean_documents(self, documents: List[Document]) -> List[Document]:
        """
        Clean a list of Document objects while preserving metadata.
        """

        cleaned_documents: List[Document] = []

        for document in documents:
            cleaned_content = self.clean_text(document.page_content)

            cleaned_document = Document(
                page_content=cleaned_content,
                metadata=document.metadata
            )

            cleaned_documents.append(cleaned_document)

        return cleaned_documents



In [6]:
cleaner = PDFTextCleaner()
pre_docs = cleaner.clean_documents(docs)

In [7]:
pre_docs

[Document(page_content='**Documentation**\n\n## **MacBook Air (13-inch, M5) - Tech Specs**\n\nYear introduced: 2026\n\n## **Finish**\n\nSky Blue Silver Starlight Midnight\n\n## **Chip**\n\n## **Apple M5 chip**\n\n10-core CPU with 4 super cores and 6 efficiency cores\n\n8-core GPU, 10-core GPU Neural Accelerators Hardware-accelerated ray tracing 16-core Neural Engine 153GB/s memory bandwidth\n\n## **Media Engine**\n\nHardware-accelerated H.264, HEVC, ProRes and ProRes RAW\n\nVideo decode engine Video encode engine ProRes encode and decode engine AV1 decode\n\n## **Configurable to:**\n\nM5 with 10-core CPU and 10-core GPU', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section=None, chunk_id=None, chunk_index=None, parent_document_id=None)),
 Document(page_content='**Memory** 16GB unified memory **Configurable to:** 24GB or 32GB **Storage** 512GB SSD **Configurable to:** 1TB, 2TB or 4TB **Display Liquid Retina display** 13.6-inch 

# Chunking

In [ ]:
from pathlib import Path
from typing import List

from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

from src.models.document import Document, DocumentMetadata


# =========================================================
# SPLITTER CONFIGURATION
# =========================================================

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3"),
    ], 
    strip_headers=False 
)


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def extract_section_name(metadata: dict) -> str | None:
    """
    Extract the most specific available markdown header.
    """

    return (
        metadata.get("header_3")
        or metadata.get("header_2")
        or metadata.get("header_1")
    )


def generate_chunk_id(
    source: str,
    chunk_index: int,
    page: int | None = None
) -> str:
    """
    Generate deterministic readable chunk IDs.
    """

    stem = Path(source).stem.lower().replace(" ", "_")

    if page is not None:
        return f"{stem}_page_{page}_chunk_{chunk_index}"

    return f"{stem}_chunk_{chunk_index}"


def create_chunk_document(
    content: str,
    original_metadata: DocumentMetadata,
    chunk_index: int,
    section: str | None = None,
) -> Document:
    """
    Create a chunked Document object with enriched metadata.
    """

    chunk_id = generate_chunk_id(
        source=original_metadata.source,
        page=original_metadata.page,
        chunk_index=chunk_index,
    )

    parent_document_id = Path(
        original_metadata.source
    ).stem.lower().replace(" ", "_")

    metadata = DocumentMetadata(
        source=original_metadata.source,
        file_type=original_metadata.file_type,
        page=original_metadata.page,
        section=section,
        chunk_id=chunk_id,
        chunk_index=chunk_index,
        parent_document_id=parent_document_id,
    )

    return Document(
        page_content=content,
        metadata=metadata,
    )


# =========================================================
# PDF / MARKDOWN CHUNKING
# =========================================================

def chunk_structured_document(document: Document) -> List[Document]:
    """
    Chunk markdown-structured documents (PDF/MD).
    """

    final_chunks = []

    # Step 1: Split by markdown headers
    header_splits = markdown_splitter.split_text(
        document.page_content
    )

    chunk_counter = 0

    for split in header_splits:

        section = extract_section_name(split.metadata)

        # Step 2: Recursively split oversized sections
        recursive_chunks = recursive_splitter.split_text(
            split.page_content
        )

        for chunk_text in recursive_chunks:

            chunk_doc = create_chunk_document(
                content=chunk_text,
                original_metadata=document.metadata,
                chunk_index=chunk_counter,
                section=section,
            )

            final_chunks.append(chunk_doc)

            chunk_counter += 1

    return final_chunks


# =========================================================
# TXT CHUNKING
# =========================================================

def chunk_text_document(document: Document) -> List[Document]:
    """
    Chunk plain text documents.
    """

    final_chunks = []

    chunks = recursive_splitter.split_text(
        document.page_content
    )

    for idx, chunk_text in enumerate(chunks):

        chunk_doc = create_chunk_document(
            content=chunk_text,
            original_metadata=document.metadata,
            chunk_index=idx,
            section=None,
        )

        final_chunks.append(chunk_doc)

    return final_chunks


# =========================================================
# MAIN DISPATCHER
# =========================================================

def chunk_documents(
    documents: List[Document]
) -> List[Document]:
    """
    Main chunking dispatcher.
    """

    all_chunks = []

    for document in documents:

        file_type = document.metadata.file_type.lower()

        # PDF + Markdown
        if file_type in ["pdf", "md", "markdown"]:

            chunks = chunk_structured_document(
                document
            )

        # TXT
        elif file_type == "txt":

            chunks = chunk_text_document(
                document
            )

        else:
            raise ValueError(
                f"Unsupported file type: {file_type}"
            )

        all_chunks.extend(chunks)

    return all_chunks



In [43]:
chunk_docs = chunk_documents(pre_docs)

In [45]:
chunk_docs[:20]

[Document(page_content='**Documentation**', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section=None, chunk_id='macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_0', chunk_index=0, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs')),
 Document(page_content='Year introduced: 2026', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section='**MacBook Air (13-inch, M5) - Tech Specs**', chunk_id='macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_1', chunk_index=1, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs')),
 Document(page_content='Sky Blue Silver Starlight Midnight', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section='**Finish**', chunk_id='macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_2', chunk_index=2, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs')),
 Document(pag

In [36]:
chunk_docs[6]

Document(page_content='**Memory** 16GB unified memory **Configurable to:** 24GB or 32GB **Storage** 512GB SSD **Configurable to:** 1TB, 2TB or 4TB **Display Liquid Retina display** 13.6-inch (diagonal) LED-backlit display with IPS technology;2 2560x1664 native resolution at 224 pixels per inch 500 nits brightness **Colour** Support for 1 billion colours Wide colour (P3) True Tone technology **Battery and** Up to 18 hours video streaming **Power** Up to 15 hours wireless web Built-in 53.8-watt-hour lithium-polymer', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=2, section=None, chunk_id='macbook_air_(13-inch,_m5)_-_tech_specs_page_2_chunk_0', chunk_index=0, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs'))

# Embeddings

In [63]:
import os
from typing import List, Dict
from dataclasses import asdict
from dotenv import load_dotenv

from google import genai
from google.genai import types

class GeminiEmbeddingGenerator:
    def __init__(self, model_name: str = "gemini-embedding-001"):
        load_dotenv()

        api_key = os.getenv("GEMINI_API_KEY")

        if not api_key:
            raise ValueError("GEMINI_API_KEY not found in environment.")

        self.client = genai.Client(api_key=api_key)

        self.model_name = model_name

    def embed_text(self, text: str) -> List[float]:
        response = self.client.models.embed_content(
            model=self.model_name,
            contents=text,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )

        return response.embedding.values

    def embed_query(self, query: str) -> List[float]:
        response = self.client.models.embed_content(
            model=self.model_name,
            contents=query,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
        )

        return response.embedding.values

    def embed_documents(self, chunks) -> List[Dict]:

        # Optimization: Extract all text blocks to perform a combined batch request
        texts_to_embed = [chunk.page_content for chunk in chunks]

        response = self.client.models.embed_content(
            model=self.model_name,
            contents=texts_to_embed,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )

        embedded_docs = []

        # When sending a list of texts, response.embeddings contains an iterable array list of records
        for idx, chunk in enumerate(chunks):
            vector = response.embeddings[idx].values

            embedded_docs.append({
                "id": chunk.metadata.chunk_id,
                "text": chunk.page_content,
                "embedding": vector,
                "metadata": asdict(chunk.metadata)
            })

        return embedded_docs

In [64]:
embedder = GeminiEmbeddingGenerator()
embedded_docs = embedder.embed_documents(chunk_docs)

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 10


In [67]:
embedded_docs[:10]

[{'id': 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_0',
  'text': '**Documentation**',
  'embedding': [-0.007151923,
   -0.010285269,
   0.016672416,
   -0.08447232,
   -0.002964101,
   0.013135681,
   -0.006744092,
   -0.011220068,
   -0.0053907502,
   -0.018202525,
   -0.024945451,
   -0.012501198,
   0.0108648995,
   0.0040892083,
   0.17420329,
   0.007141384,
   0.0015320816,
   0.0050679827,
   -0.0053384146,
   -0.019643983,
   0.0008554697,
   0.0009863783,
   0.0075693903,
   -0.0073781745,
   -0.0017714066,
   -0.012086095,
   0.012682609,
   0.0035723594,
   0.015909858,
   0.009759293,
   -0.0034338802,
   0.002891595,
   -0.0010482256,
   0.01564127,
   -0.0056658033,
   0.018811774,
   0.0059727915,
   -0.02122512,
   -0.00026916884,
   0.0033773466,
   -0.00654934,
   -0.0016830051,
   0.0016706015,
   -0.0008481072,
   0.0028876332,
   0.0041687125,
   -0.012793235,
   -0.0039184643,
   -0.013584178,
   0.024338873,
   -0.0015350458,
   -0.006865194,
   -0.0034

# Vector Database